A demonstration to calculate the Potential Energy Anomaly for Profile data.


### Relevant imports and filepath configuration

In [1]:
import os
os.chdir('../../../../')
import coast
import numpy as np
from os import path
import matplotlib.pyplot as plt
import matplotlib.colors as colors  # colormap fiddling

In [2]:
# set some paths
root = "./"
dn_files = root + "./example_files/"
fn_prof = path.join(dn_files, "coast_example_en4_201008.nc")
fn_cfg_prof = path.join("config","example_en4_profiles.json")

### Loading data

In [3]:
# Create a Profile object and load in the data:
profile = coast.Profile(config=fn_cfg_prof)
profile.read_en4( fn_prof )

config/example_en4_profiles.json


FileNotFoundError: [Errno 2] No such file or directory: b'/home/users/jholt/Git/COAsT/example_files/coast_example_en4_201008.nc'

If you are using EN4 data, you can use the process_en4() routine to apply quality control flags to the data (replacing with NaNs):

In [ ]:
processed_profile = profile.process_en4()
profile = processed_profile

### Inspect profile locations
Have a look inside the `profile.py` class to see what it can do. But first have a look at the spatial distribution of profiles.

In [ ]:
profile.plot_map()

### Calculates Potential Energy Anomaly

Similar to the Gridded object, potential energy anomaly can be calculated for Profile objects. This method exists within a `ProfileStratifiction` object, which must be initialised



In [ ]:
pa = coast.ProfileStratification(profile)

Define a gridded object to supply the bathymetry

In [ ]:
fn_nemo_dom = dn_files + "coast_example_nemo_domain.nc"
config_t = root + "./config/example_nemo_grid_t.json"
nemo = coast.Gridded(fn_domain=fn_nemo_dom, config=config_t)

Potential energy anomaly is calculated to a prescribed depth, Zmax:

In [ ]:
Zmax = 200  # metres
pa.calc_pea(profile, nemo, Zmax)

In this calculation a number of steps happen within ProfileStratification: for a supplied Profile, first the vertical spacing is calculated

``profile.calculate_vertical_spacing()``

Then a depth mask is calculated to exclude depth below the Zmax threshold.
(The last depth level is a float between 0,1 denoting how much of the next spacing below is deeper than Zmax - To facilitate the integral to Zmax)

``Zd_mask, kmax = profile.calculate_vertical_mask(Zmax)``

Then densities (depth varying and depth averaged) are computed from the temperature and salinity fields
``profile.construct_density()``

Finally the depth integrals are calculated.


## Make a plot


THERE IS OBVIOUSLY AN ISSUE HERE WITH NEGATIVE PEA VALUES AND SMALL POSITIVE VALUES...

In [ ]:
fig, ax = pa.quick_plot("pea")
fig.tight_layout()

In [ ]:
plt.scatter( pa.dataset.longitude,
            pa.dataset.latitude,
            s=4, c=pa.dataset.pea)
plt.clim([0,10])
plt.colorbar()